# 05 — Governed Boundary Contracts

This notebook demonstrates how to declare and enforce **governed portal
contracts**: the ontology, ownership, classification, and alignment
metadata that gives a boundary its authority, not just its structural shape.

SHACL shapes validate *what* can cross a membrane. This notebook shows
how to declare *under whose authority*, *governed by which ontology*,
*conforming to which standard*, and *at what classification level* the
data crosses.

Scenario: a university has two departments. The Registrar uses CCO
to model student records. The Career Services office uses Schema.org
for its public-facing job placement directory. A governed portal
translates CCO student records into Schema.org profiles, with an
alignment holon documenting the vocabulary mapping and its provenance.

In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()

# ── Organization root ──

ds.add_holon("urn:holon:university", "Greenfield University")

# ── Data domains (governance scope) ──

ds.add_holon("urn:holon:domain-registrar", "Registrar Domain",
             member_of="urn:holon:university")
ds.add_interior("urn:holon:domain-registrar", '''
    @prefix cga:    <urn:holonic:ontology:> .
    @prefix schema: <https://schema.org/> .

    <urn:domain:registrar> a cga:DataDomain ;
        rdfs:label "Student Records" ;
        cga:domainSteward <urn:person:j-chen> ;
        cga:domainPolicy  <urn:policy:ferpa-compliance> .

    <urn:person:j-chen> a schema:Person ;
        schema:name "J. Chen" ;
        schema:jobTitle "Registrar Data Steward" .
''')

ds.add_holon("urn:holon:domain-careers", "Career Services Domain",
             member_of="urn:holon:university")
ds.add_interior("urn:holon:domain-careers", '''
    @prefix cga:    <urn:holonic:ontology:> .
    @prefix schema: <https://schema.org/> .

    <urn:domain:careers> a cga:DataDomain ;
        rdfs:label "Career Placement" ;
        cga:domainSteward <urn:person:m-park> ;
        cga:domainPolicy  <urn:policy:career-data-sharing> .

    <urn:person:m-park> a schema:Person ;
        schema:name "M. Park" ;
        schema:jobTitle "Career Services Data Steward" .
''')

print(f"Holons: {len(ds.list_holons())}")

## Department holons with governance metadata

Each department declares which ontology governs its interior data,
who stewards it, and what classification level its data carries.

In [ ]:
# ── Registrar department (CCO-governed, PII data) ──

ds.add_holon("urn:holon:registrar", "Registrar Office",
             member_of="urn:holon:domain-registrar")
ds.add_interior("urn:holon:registrar", '''
    @prefix cga:     <urn:holonic:ontology:> .
    @prefix schema:  <https://schema.org/> .
    @prefix dcterms: <http://purl.org/dc/terms/> .

    <urn:holon:registrar> cga:dataSteward <urn:person:j-chen> ;
        cga:dataOwner          <urn:person:a-kim> ;
        cga:dataClassification cga:PII ;
        cga:sourceOfTruthFor   <urn:domain:registrar> ;
        dcterms:conformsTo     <https://www.commoncoreontologies.org/CommonCoreOntologiesMerged> .

    <urn:person:a-kim> a schema:Person ;
        schema:name "A. Kim" ;
        schema:jobTitle "University Registrar" .
''')

# Student records in CCO vocabulary
ds.add_interior("urn:holon:registrar", '''
    @prefix cco: <https://www.commoncoreontologies.org/> .

    <urn:student:001> a cco:ont00001262 ;       # cco:Person
        rdfs:label "Dana Rivera" ;
        cco:ont00001765 "dana.rivera@greenfield.edu" .  # has_text_value

    <urn:name:001> a cco:ont00000003 ;           # cco:DesignativeName
        cco:ont00001765 "Dana Rivera" .

    <urn:role:001> a cco:ont00000984 ;           # cco:OccupationRole
        rdfs:label "Computer Science, BS 2026" .

    <urn:student:002> a cco:ont00001262 ;        # cco:Person
        rdfs:label "Sam Okafor" ;
        cco:ont00001765 "sam.okafor@greenfield.edu" .

    <urn:name:002> a cco:ont00000003 ;
        cco:ont00001765 "Sam Okafor" .

    <urn:role:002> a cco:ont00000984 ;
        rdfs:label "Data Science, MS 2025" .
''', graph_iri="urn:holon:registrar/interior/students")

# ── Career Services department (Schema.org-governed, public data) ──

ds.add_holon("urn:holon:careers", "Career Services",
             member_of="urn:holon:domain-careers")
ds.add_interior("urn:holon:careers", '''
    @prefix cga:     <urn:holonic:ontology:> .
    @prefix schema:  <https://schema.org/> .
    @prefix dcterms: <http://purl.org/dc/terms/> .

    <urn:holon:careers> cga:dataSteward <urn:person:m-park> ;
        cga:dataOwner          <urn:person:r-jones> ;
        cga:dataClassification cga:Public ;
        cga:repositoryOfTruthFor <urn:domain:careers> ;
        dcterms:conformsTo     <https://schema.org/> .

    <urn:person:r-jones> a schema:Person ;
        schema:name "R. Jones" ;
        schema:jobTitle "Director of Career Services" .
''')

print(f"Holons: {len(ds.list_holons())}")
print(f"Registrar classification: cga:PII")
print(f"Careers classification:   cga:Public")

## Alignment Holon

An `AlignmentHolon` contains the vocabulary mapping between CCO and
Schema.org. It exists as a first-class entity so the mapping itself
has provenance, ownership, and versioning.

In [ ]:
ds.add_holon("urn:holon:alignment-cco-schema", "CCO to Schema.org Alignment",
             member_of="urn:holon:university")
ds.add_interior("urn:holon:alignment-cco-schema", '''
    @prefix cga:     <urn:holonic:ontology:> .
    @prefix dcterms: <http://purl.org/dc/terms/> .
    @prefix skos:    <http://www.w3.org/2004/02/skos/core#> .

    <urn:alignment:cco-to-schema> a cga:AlignmentHolon ;
        rdfs:label "CCO Person to Schema.org Person" ;
        dcterms:creator    <urn:person:j-chen> ;
        dcterms:modified   "2026-04-01"^^<http://www.w3.org/2001/XMLSchema#date> ;
        skos:note "Maps CCO Person/DesignativeName/OccupationRole to Schema.org Person with name, email, and description." .
''')

print("Alignment holon created")

## The governed portal

The portal carries the CONSTRUCT query and a `cga:usesAlignment`
link to the alignment holon. The classification boundary crossing
(PII source to Public target) will be flagged by
`PortalClassificationShape`.

In [ ]:
ds.add_portal(
    "urn:portal:registrar-to-careers",
    source_iri="urn:holon:registrar",
    target_iri="urn:holon:careers",
    construct_query='''
        PREFIX cco:    <https://www.commoncoreontologies.org/>
        PREFIX schema: <https://schema.org/>

        CONSTRUCT {
            ?profile a schema:Person ;
                schema:name  ?name ;
                schema:email ?email ;
                schema:description ?program .
        }
        WHERE {
            GRAPH ?g {
                ?student a cco:ont00001262 ;         # Person
                    cco:ont00001765 ?email .          # has_text_value

                ?nameEntity a cco:ont00000003 ;      # DesignativeName
                    cco:ont00001765 ?name .

                ?role a cco:ont00000984 ;            # OccupationRole
                    rdfs:label ?program .

                FILTER(CONTAINS(STR(?nameEntity), REPLACE(STR(?student), "urn:student:", "urn:name:")))
                FILTER(CONTAINS(STR(?role), REPLACE(STR(?student), "urn:student:", "urn:role:")))
            }
            BIND(IRI(CONCAT("urn:profile:", ENCODE_FOR_URI(?name))) AS ?profile)
        }
    ''',
    extra_ttl='''
        @prefix cga: <urn:holonic:ontology:> .
        <urn:portal:registrar-to-careers> cga:usesAlignment
            <urn:holon:alignment-cco-schema> .
    ''',
    label="Registrar to Careers (CCO to Schema.org)",
)

# Verify the alignment link
rows = list(ds.query('''
    PREFIX cga: <urn:holonic:ontology:>
    SELECT ?alignment WHERE {
        GRAPH ?g {
            <urn:portal:registrar-to-careers> cga:usesAlignment ?alignment .
        }
    }
'''))
print(f"Portal alignment: {rows[0]['alignment']}")

## Traversal with governance trail

In [ ]:
projected, membrane = ds.traverse(
    source_iri="urn:holon:registrar",
    target_iri="urn:holon:careers",
    validate=True,
)

print(f"Projected {len(projected)} triples")
print(f"Membrane: {membrane.health.value}")
print()
for s, p, o in sorted(projected):
    print(f"  {s.split('/')[-1]:25s}  {p.split('/')[-1]:15s}  {o}")

In [ ]:
# Query the full governance chain
governance = list(ds.query('''
    PREFIX cga:     <urn:holonic:ontology:>
    PREFIX schema:  <https://schema.org/>

    SELECT DISTINCT ?steward_name ?classification ?alignment_label
    WHERE {
        GRAPH ?g1 {
            <urn:holon:registrar> cga:dataSteward ?steward .
            <urn:holon:registrar> cga:dataClassification ?classification .
        }
        GRAPH ?g2 { <urn:portal:registrar-to-careers> cga:usesAlignment ?alignment . }
        GRAPH ?g3 { ?steward schema:name ?steward_name . }
        GRAPH ?g4 { ?alignment rdfs:label ?alignment_label . }
    }
'''))

print("Governance chain for Registrar to Careers portal:")
for row in governance:
    print(f"  Steward:        {row['steward_name']}")
    print(f"  Classification: {row['classification']}")
    print(f"  Alignment:      {row['alignment_label']}")

## What this demonstrates

The structural boundary (SHACL shapes) validates *what* can cross.
The governance metadata answers the questions underneath:

- **Which ontology governs each side?** `dcterms:conformsTo` on each
  department (CCO for the Registrar, Schema.org for Career Services).
- **Who owns the boundary contract?** `cga:dataSteward` and
  `cga:dataOwner` on each holon.
- **What alignment justifies the translation?** `cga:usesAlignment`
  on the portal links to the alignment holon.
- **What classification constraints apply?** The Registrar holds PII
  (student records under FERPA); Career Services is Public. The
  `PortalClassificationShape` would flag this boundary crossing,
  surfacing the need for a data-sharing agreement.

See also:

- `04_cco_to_schemaorg` — the translation mechanics without
  governance context
- `docs/source/ontology.md` — the full CGA governance vocabulary